In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from scipy.stats import norm

In [2]:
data = pd.read_csv("train.csv")
data['date'] = pd.to_datetime(data['date'])

df = data[(data['store']==1) & (data['item']==1)].copy()
df = df.sort_values('date')

df['lag_1'] = df['sales'].shift(1)
df['lag_7'] = df['sales'].shift(7)
df['lag_30'] = df['sales'].shift(30)

df['rolling_7'] = df['sales'].shift(1).rolling(7).mean()
df['rolling_30'] = df['sales'].shift(1).rolling(30).mean()

df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek

df = df.dropna()

features = ['lag_1','lag_7','lag_30','rolling_7','rolling_30','month','day_of_week']

split = int(len(df)*0.8)

train = df.iloc[:split]
valid = df.iloc[split:]

X_train = train[features]
y_train = train['sales']

X_valid = valid[features]
y_valid = valid['sales']

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(X_train,y_train)

predictions = model.predict(X_valid)

In [3]:
forecast_mean = np.mean(predictions)

residuals = y_valid - predictions
forecast_std = np.std(residuals)

print("Average predicted daily demand:",forecast_mean)
print("Demand uncertainty (std):",forecast_std)

Average predicted daily demand: 21.362328
Demand uncertainty (std): 5.174925227010631


In [4]:
lead_time_days = 7
service_level = 0.95

z_score = norm.ppf(service_level)

### Safety Stock


In [5]:
safety_stock = z_score * forecast_std * np.sqrt(lead_time_days)

print("Safety Stock:",safety_stock)

Safety Stock: 22.520620684482438


### Reorder Point

In [6]:
reorder_point = forecast_mean * lead_time_days + safety_stock

print("Reorder Point:",reorder_point)

Reorder Point: 172.05690608487305


### Stock-Out Risk

In [7]:
current_inventory = 150

expected_demand = forecast_mean * lead_time_days
demand_std = forecast_std * np.sqrt(lead_time_days)

stockout_probability = 1 - norm.cdf(
    current_inventory,
    loc=expected_demand,
    scale=demand_std
)

print("Stockout Probability:",stockout_probability)

Stockout Probability: 0.48649095296219746
